# Activation Patching

TODO: Add desc

## Inital Setup

In [1]:
# For loading HF_Token
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [2]:
# Import stuff
import torch
import torch.nn as nn
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from jaxtyping import Float
from functools import partial

c:\Users\finla\Documents\work\mech-interp-mini-projects\ioi-activation-patching\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Setting plotly renderer to work for notebooks
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [4]:
import circuitsvis as cv

# Testing that the library works
cv.examples.hello("Fin")

In [5]:
# import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import FactoredMatrix, HookedTransformer
from transformer_lens.model_bridge import TransformerBridge

In [6]:
# Turn off auto differentation to save memory
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [7]:
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)


def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(y=utils.to_numpy(tensor), labels={"x": xaxis, "y": yaxis}, **kwargs).show(renderer)


def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(y=y, x=x, labels={"x": xaxis, "y": yaxis, "color": caxis}, **kwargs).show(renderer)

## Activation Patching Code

In [8]:
device = utils.get_device()

In [9]:
model = TransformerBridge.boot_transformers("openai-community/gpt2", device=device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6645.94it/s]


In [10]:
def logits_to_logit_diff(logits, correct_answer=" Jill", incorrect_answer=" Jack"):
    correct_index = model.to_single_token(correct_answer)
    incorrect_index = model.to_single_token(incorrect_answer)
    return logits[0, -1, correct_index] - logits[0, -1, incorrect_index]

In [11]:
clean_input = "Jack and Jill went on a date, Jack gave flowers to"
corrupted_input = "Jack and Jill went on a date, Jill gave flowers to"

clean_tokens = model.to_tokens(clean_input)
clean_logits, clean_cache = model.run_with_cache(clean_input)
clean_logit_diff = logits_to_logit_diff(clean_logits)


corrupted_tokens = model.to_tokens(corrupted_input)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_input)
corrupted_logit_diff = logits_to_logit_diff(corrupted_logits)

print(clean_logit_diff)
print(corrupted_logit_diff)


tensor(4.4306)
tensor(-5.6188)


Now we must do the 'patching', we have to get the activations from the corrupted input and patch them into the clean input and then see how the final logit differs.

for every atten head
    replace clean with corrupted atten

In [18]:
from functools import partial
from transformer_lens import utils

def activation_patching_hook(value: Float[torch.Tensor, "batch pos head d_head"], hook, head=None):
    clean_head = clean_cache[hook.name]
    value[:, :, head, :] = clean_head[:, :, head, :]
    return value

results = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

for layer in tqdm.tqdm(range(model.cfg.n_layers)):
    for head in range(model.cfg.n_heads):
        hook_fn = partial(activation_patching_hook, head=head)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("z", layer), hook_fn)]
        )

        patched_logit_diff = logits_to_logit_diff(patched_logits)
        results[layer, head] = (patched_logit_diff - corrupted_logit_diff) / (
            clean_logit_diff - corrupted_logit_diff
        )

100%|██████████| 12/12 [00:09<00:00,  1.29it/s]


In [19]:
imshow(
    results,
    xaxis="Head",
    yaxis="Layer",
    title="Activation Patching: Loss by Layer and Head",
)

In [14]:
from functools import partial
from transformer_lens import utils

def activation_patching_hook(value: Float[torch.Tensor, "batch pos head d_head"], hook, head=None):
    corrupted_head = corrupted_cache[hook.name]
    value[:, :, head, :] = corrupted_head[:, :, head, :]
    return value

results = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

for layer in tqdm.tqdm(range(model.cfg.n_layers)):
    for head in range(model.cfg.n_heads):
        hook_fn = partial(activation_patching_hook, head=head)
        patched_logits = model.run_with_hooks(
            clean_tokens,
            fwd_hooks=[(utils.get_act_name("z", layer), hook_fn)]
        )

        patched_logit_diff = logits_to_logit_diff(patched_logits)
        results[layer, head] = (clean_logit_diff - patched_logit_diff) / (
            clean_logit_diff - corrupted_logit_diff
        )

100%|██████████| 12/12 [00:10<00:00,  1.13it/s]


In [15]:
imshow(
    results,
    xaxis="Head",
    yaxis="Layer",
    title="Activation Patching: Loss by Layer and Head",
)

In [16]:
def layer_patching_hook(value: Float[torch.Tensor, "batch, pos, d_model"], hook, token_pos=None):
    clean_head = clean_cache[hook.name]
    value[:, token_pos, :] = clean_head[:, token_pos, :]
    return value

a = utils.get_act_name("resid_pre", 1)
clean_cache[a].shape # batch pos d_model

len_tokens = len(corrupted_tokens[0])

results = torch.zeros(model.cfg.n_layers, len_tokens)

for layer in tqdm.tqdm(range(model.cfg.n_layers)):
    for token_pos in range(len_tokens):
        hook_fn = partial(layer_patching_hook, token_pos=token_pos)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("resid_pre", layer), hook_fn)]
        )
        
        patched_logit_diff = logits_to_logit_diff(patched_logits)
        results[layer, token_pos] = (patched_logit_diff - corrupted_logit_diff) / (
            clean_logit_diff - corrupted_logit_diff
        )

100%|██████████| 12/12 [00:11<00:00,  1.08it/s]


In [17]:
imshow(
    results,
    xaxis="Token",
    yaxis="Layer",
    title="Activation Patching: Loss by Layer and Token",
)